In [ ]:
import zipfile
with zipfile.ZipFile('ml-latest-small.zip', 'r') as zip_ref:
  zip_ref.extractall('data')


In [ ]:
# import the dataset
import pandas as pd
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

In [ ]:
print('The dimensions of movies dataframe are:', movies_df.shape, '\nThe dimensions of ratings dataframe are:', ratings_df.shape)
print('')

The dimensions of movies dataframe are: (9742, 3) 
The dimensions of ratings dataframe are: (100836, 4)



In [ ]:
# Take a look at movies dataframe
movies_df.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
# Take a look at ratings dataframe
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
# Movie ID to movie name mapping
movie_names = movies_df.set_index('movieId')['title'].to_dict()
# Count unique users and movies
n_users = len(ratings_df.userId.unique())
n_items = len(ratings_df.movieId.unique())
print("Number of unique users:", n_users)
print("Number of unique movies:", n_items)
# Total matrix size
print("The full rating matrix will have:", n_users * n_items, "elements.")
print('---')
# Number of ratings
print("Number of ratings:", len(ratings_df))
print("Therefore: ", len(ratings_df) / (n_users*n_items) * 100, '% of the matrix is filled.')
print("We have an incredibly sparse matrix to work with here.")
print("And ... as you can imagine, as the number of users and products grow, the number of elements will increase by n*2")
print("You are going to need a lot of memory to work with global scale ... storing a full matrix in memory would be a challenge.")
print("One advantage here is that matrix factorization can realize the rating matrix implicitly, thus we don't need all the data")


Number of unique users: 610
Number of unique movies: 9724
The full rating matrix will have: 5931640 elements.
---
Number of ratings: 100836
Therefore:  1.6999683055613624 % of the matrix is filled.
We have an incredibly sparse matrix to work with here.
And ... as you can imagine, as the number of users and products grow, the number of elements will increase by n*2
You are going to need a lot of memory to work with global scale ... storing a full matrix in memory would be a challenge.
One advantage here is that matrix factorization can realize the rating matrix implicitly, thus we don't need all the data


In [ ]:
import torch
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader

# Creating the dataloader (necessary for PyTorch)

class Loader(Dataset):

    def __init__(self):
        self.ratings = ratings_df.copy()

        # Extract all user IDs and movie IDs
        users = ratings_df.userId.unique()
        movies = ratings_df.movieId.unique()

        # Producing new continuous IDs for users and movies
        self.userid2idx = {o: i for i, o in enumerate(users)}
        self.movieid2idx = {o: i for i, o in enumerate(movies)}

        # Reverse mappings
        self.idx2userid = {i: o for o, i in self.userid2idx.items()}
        self.idx2movieid = {i: o for o, i in self.movieid2idx.items()}

        # Replace original IDs with indexed IDs
        self.ratings.movieId = ratings_df.movieId.apply(
            lambda x: self.movieid2idx[x]
        )

        self.ratings.userId = ratings_df.userId.apply(
            lambda x: self.userid2idx[x]
        )

        # Features and targets
        self.x = self.ratings.drop(
            ['rating', 'timestamp'],
            axis=1
        ).values

        self.y = self.ratings['rating'].values

        # Transforms the data to tensors (ready for torch models.)
        self.x = torch.tensor(self.x)
        self.y = torch.tensor(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.ratings)

In [ ]:
import torch
import torch.nn as nn

class MatrixFactorization(nn.Module):

    def __init__(self, n_users, n_items, n_factors=20):
        super().__init__()

        # User embeddings
        self.user_factors = nn.Embedding(
            n_users,
            n_factors
        )

        # Movie embeddings
        self.item_factors = nn.Embedding(
            n_items,
            n_factors
        )

        # Initialize weights
        self.user_factors.weight.data.uniform_(0, 0.05)
        self.item_factors.weight.data.uniform_(0, 0.05)

    def forward(self, data):

        # Extract user and movie IDs
        users = data[:, 0]
        items = data[:, 1]

        # Get embeddings
        user_embedding = self.user_factors(users)
        item_embedding = self.item_factors(items)

        # Dot product
        predictions = (
            user_embedding * item_embedding
        ).sum(1)

        return predictions

In [ ]:
num_epochs = 128

# Check if GPU is available
cuda = torch.cuda.is_available()

print("Is running on GPU:", cuda)

# Initialize model
model = MatrixFactorization(
    n_users,
    n_items,
    n_factors=8
)

print(model)

# Print model parameters
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.data)

# Move model to GPU if available
if cuda:
    model = model.cuda()

# MSE Loss
loss_fn = torch.nn.MSELoss()

# Adam Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# Training data
train_set = Loader()

train_loader = DataLoader(
    train_set,
    batch_size=128,
    shuffle=True
)

Is running on GPU: False
MatrixFactorization(
  (user_factors): Embedding(610, 8)
  (item_factors): Embedding(9724, 8)
)
user_factors.weight tensor([[0.0326, 0.0437, 0.0335,  ..., 0.0003, 0.0027, 0.0463],
        [0.0284, 0.0367, 0.0147,  ..., 0.0485, 0.0377, 0.0386],
        [0.0298, 0.0159, 0.0340,  ..., 0.0143, 0.0053, 0.0287],
        ...,
        [0.0384, 0.0102, 0.0020,  ..., 0.0288, 0.0470, 0.0047],
        [0.0186, 0.0286, 0.0398,  ..., 0.0052, 0.0050, 0.0119],
        [0.0112, 0.0316, 0.0359,  ..., 0.0262, 0.0287, 0.0318]])
item_factors.weight tensor([[0.0349, 0.0198, 0.0012,  ..., 0.0217, 0.0060, 0.0285],
        [0.0367, 0.0394, 0.0476,  ..., 0.0268, 0.0430, 0.0328],
        [0.0489, 0.0493, 0.0351,  ..., 0.0230, 0.0103, 0.0485],
        ...,
        [0.0463, 0.0259, 0.0384,  ..., 0.0369, 0.0002, 0.0391],
        [0.0345, 0.0319, 0.0471,  ..., 0.0406, 0.0130, 0.0309],
        [0.0488, 0.0030, 0.0193,  ..., 0.0213, 0.0362, 0.0158]])


In [ ]:
from tqdm.notebook import tqdm

# Training loop
for it in tqdm(range(num_epochs)):

    losses = []

    for x, y in train_loader:

        # Move data to GPU
        if cuda:
            x = x.cuda()
            y = y.cuda()

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(x)

        # Calculate loss
        loss = loss_fn(
            outputs.squeeze(),
            y.type(torch.float32)
        )

        # Store loss
        losses.append(loss.item())

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

    # Print average loss per epoch
    print(
        "iter #{}".format(it),
        "Loss:",
        sum(losses) / len(losses)
    )

  0%|          | 0/128 [00:00<?, ?it/s]

iter #0 Loss: 11.06140935965601
iter #1 Loss: 4.731660937899865
iter #2 Loss: 2.4678415483629643
iter #3 Loss: 1.7176357791357233
iter #4 Loss: 1.3432625749994656
iter #5 Loss: 1.1270117180299033
iter #6 Loss: 0.9907731857094063
iter #7 Loss: 0.8997445050532442
iter #8 Loss: 0.836745453364958
iter #9 Loss: 0.7920654949349195
iter #10 Loss: 0.7590997102221256
iter #11 Loss: 0.7344923191869319
iter #12 Loss: 0.7161477155945628
iter #13 Loss: 0.7017374672184741
iter #14 Loss: 0.6906606945667775
iter #15 Loss: 0.6815442097217298
iter #16 Loss: 0.6750228393652717
iter #17 Loss: 0.6697985050339385
iter #18 Loss: 0.6657725853968393
iter #19 Loss: 0.6629669072513048
iter #20 Loss: 0.6604916188998271
iter #21 Loss: 0.6588464034768531
iter #22 Loss: 0.6579022832252652
iter #23 Loss: 0.6567527316321576
iter #24 Loss: 0.6559626180584055
iter #25 Loss: 0.655054506689764
iter #26 Loss: 0.6544344531854397
iter #27 Loss: 0.6537760634183278
iter #28 Loss: 0.6527520278610554
iter #29 Loss: 0.65141092456

In [ ]:
# By training the model, we will have tuned latent factors for movies and users.

c = 0
uw = 0
iw = 0

for name, param in model.named_parameters():

    if param.requires_grad:

        print(name, param.data)

        if c == 0:
            uw = param.data
            c += 1

        else:
            iw = param.data

        # print('param_data', param.data)

user_factors.weight tensor([[ 1.0954,  1.2614,  1.1111,  ...,  1.7285,  1.0175,  1.1054],
        [ 0.8659,  1.9309,  1.1902,  ...,  1.5735,  0.4449,  1.7081],
        [-1.3746,  2.3986,  1.3548,  ..., -1.4414, -1.1026,  1.6846],
        ...,
        [ 2.1009,  1.2336,  0.8193,  ...,  0.8231,  2.3514, -0.3148],
        [ 1.3417,  0.2344,  0.6973,  ...,  1.0721,  1.1769,  1.1191],
        [ 0.0648,  1.2311,  1.4418,  ...,  0.7934,  1.3701,  1.9080]])
item_factors.weight tensor([[0.4118, 0.3889, 0.5941,  ..., 0.4727, 0.2665, 0.6791],
        [0.4714, 0.1093, 0.5742,  ..., 0.3149, 0.0959, 0.5971],
        [0.5486, 0.2680, 0.5165,  ..., 0.1696, 0.7609, 0.5491],
        ...,
        [0.3408, 0.3516, 0.3638,  ..., 0.3637, 0.3253, 0.3640],
        [0.2765, 0.3851, 0.3996,  ..., 0.3919, 0.3670, 0.3881],
        [0.3421, 0.3804, 0.3973,  ..., 0.3967, 0.4149, 0.3967]])


In [ ]:
trained_movie_embeddings = model.item_factors.weight.data.cpu().numpy()

In [ ]:
trained_movie_embeddings # unique movie factor weights

array([[0.41178024, 0.3889498 , 0.5940676 , ..., 0.47274157, 0.26645875,
        0.6790804 ],
       [0.47135866, 0.10934938, 0.57424325, ..., 0.3149494 , 0.09593393,
        0.5970567 ],
       [0.5485605 , 0.2679667 , 0.5165235 , ..., 0.16961893, 0.7609192 ,
        0.5490744 ],
       ...,
       [0.3408253 , 0.3516395 , 0.36378184, ..., 0.3637068 , 0.32526997,
        0.3639856 ],
       [0.27653575, 0.38510796, 0.39962944, ..., 0.39185712, 0.36701274,
        0.3881183 ],
       [0.3421454 , 0.38040233, 0.3973211 , ..., 0.39669517, 0.41490203,
        0.396689  ]], shape=(9724, 8), dtype=float32)

In [35]:
from sklearn.cluster import KMeans
# Fit the clusters based on the movie weights
kmeans = KMeans(n_clusters=10, random_state=0).fit(trained_movie_embeddings)

In [36]:
"""
It can be seen here that the movies in the same cluster tend to have
similar genres. Also note that the algorithm is unfamiliar with the movie names
and only obtained the relationships by looking at the numbers
representing how users responded to movie selections.
"""

import numpy as np

for cluster in range(10):

    print(f"\nCluster #{cluster}")

    movs = []

    for movidx in np.where(kmeans.labels_ == cluster)[0]:

        movid = train_set.idx2movieid[movidx]

        rat_count = ratings_df.loc[
            ratings_df['movieId'] == movid
        ].shape[0]

        movs.append(
            (movie_names[movid], rat_count)
        )

    for mov in sorted(
        movs,
        key=lambda tup: tup[1],
        reverse=True
    )[:10]:

        print("\t", mov[0])


Cluster #0
	 Schindler's List (1993)
	 Dances with Wolves (1990)
	 Titanic (1997)
	 Sleepless in Seattle (1993)
	 Casablanca (1942)
	 As Good as It Gets (1997)
	 Wizard of Oz, The (1939)
	 Dead Poets Society (1989)
	 High Fidelity (2000)
	 Bourne Supremacy, The (2004)

Cluster #1
	 Pulp Fiction (1994)
	 Fight Club (1999)
	 Usual Suspects, The (1995)
	 American Beauty (1999)
	 Seven (a.k.a. Se7en) (1995)
	 Godfather, The (1972)
	 Fargo (1996)
	 Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
	 Memento (2000)
	 Inception (2010)

Cluster #2
	 Forrest Gump (1994)
	 Shawshank Redemption, The (1994)
	 Silence of the Lambs, The (1991)
	 Matrix, The (1999)
	 Braveheart (1995)
	 Terminator 2: Judgment Day (1991)
	 Toy Story (1995)
	 Star Wars: Episode V - The Empire Strikes Back (1980)
	 Apollo 13 (1995)
	 Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)

Cluster #3
	 Jurassic Park (1993)
	 Independence Day (a.k.a. ID4) (1996)
	 True Lies (1994)
	 Mask, The (1994)
	 Mrs